# 🦀 Day 3: Debugging Macros
---

Today we learn how to **debug macros** — because macro errors can be cryptic!

- **Why macro debugging is hard**: Errors point to expanded code, not your macro
- **cargo expand**: See exactly what macros generate
- **trace_macros!** and **log_syntax!** (nightly features)
- **Step-by-step expansion**: Manual walkthrough
- **Common errors**: Ambiguity, unexpected tokens
- **Best practices**: Start simple, test incrementally

## Why Macro Debugging Is Hard

When a macro fails, the compiler reports errors in the **expanded** code, not in your macro definition. The line numbers and snippets often don't match what you wrote. Tools like `cargo expand` help you see the actual output.

In [ ]:
// Our hashmap! macro — let's trace what it expands to
use std::collections::HashMap;

macro_rules! hashmap {
    ($( $key:expr => $val:expr ),* $(,)?) => {
        {
            let mut m = HashMap::new();
            $( m.insert($key, $val); )*
            m
        }
    };
}

let m = hashmap! { "x" => 10, "y" => 20 };
println!("{:?}", m);

// Manual expansion of hashmap! { "x" => 10, "y" => 20 }:
// {
//     let mut m = HashMap::new();
//     m.insert("x", 10);
//     m.insert("y", 20);
//     m
// }

In [ ]:
// my_vec! expansion by hand
macro_rules! my_vec {
    () => { Vec::new() };
    ($($x:expr),* $(,)?) => {
        { let mut v = Vec::new(); $( v.push($x); )* v }
    };
}

let v = my_vec![1, 2, 3];
// Expands to: { let mut v = Vec::new(); v.push(1); v.push(2); v.push(3); v }
println!("{:?}", v);

## cargo expand (outside EvCxR)

In a Cargo project, run:
```bash
cargo install cargo-expand
cargo expand
```

This prints the fully expanded source. For `trace_macros!(true)` and `log_syntax!`, you need a nightly compiler — they print expansion steps to stderr.

In [ ]:
// Common macro errors: ambiguity
// BAD: ($x:expr $y:expr) can match "1 2" as one expr or two
// FIX: Use clearer separators: ($x:expr, $y:expr)

// Unexpected token: wrong fragment type
macro_rules! bad_example {
    ($x:ident) => { $x + 1 };  // Fails if you pass 5 (a literal, not ident)
}

let a = 10;
println!("bad_example!(a) = {}", bad_example!(a));  // Works: a is ident
// bad_example!(5);  // Would fail: 5 is not an ident

## 📝 Exercise: Debug a broken macro

The macro below has bugs. Fix it so `fixed_double!(5)` prints `10` and `fixed_double!(3 + 2)` prints `10`.

In [ ]:
// BROKEN — fix it! (hint: wrong variable in expansion)
macro_rules! fixed_double {
    ($x:expr) => {
        $y * 2  // BUG: should be $x, not $y
    };
}

// YOUR CODE HERE: fix the macro, then uncomment and run:
// println!("{}", fixed_double!(5));
// println!("{}", fixed_double!(3 + 2));

## 🎯 Key Takeaways

1. Macro errors point to **expanded** code — use `cargo expand` to see it
2. **trace_macros!(true)** and **log_syntax!** (nightly) help trace expansion
3. **Manual expansion**: Substitute matchers into the transcriber step-by-step
4. **Ambiguity errors**: Use clear separators (e.g. `,`) between fragments
5. **Unexpected token**: Match the right fragment type (`expr` vs `ident` vs `ty`)
6. **Best practice**: Start with a minimal macro, add complexity incrementally, test often

---
**Tomorrow:** Procedural macros — intro and types